In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv(override=True)
import json
import psutil





In [3]:
key = os.getenv("OPENAI_API_KEY")
print("url:", os.getenv("OPENAI_BASE_URL"))
print("prefix:", (key or "")[:7], "len:", len(key or ""))

url: https://api.gapgpt.app/v1
prefix: sk-B7eR len: 51


In [4]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)

In [5]:
def get_system_stats():
    cpu_percent = psutil.cpu_percent(interval=1)
    ram_percent = psutil.virtual_memory().percent
    stats = {"cpu":cpu_percent, "ram":ram_percent}
    return json.dumps(stats)
print(get_system_stats())

{"cpu": 9.6, "ram": 50.2}


In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_system_stats",
            "description": "وضعیت زنده درصد مصرف پردازنده (CPU) و رم (RAM) سیستم اوبونتو را برمی‌گرداند.",
            "parameters": {
                "type": "object",
                "properties": {},  # خالی است چون این تابع هیچ ورودی‌ای لازم ندارد
                "required": [],
            },
        },
    }
]


In [7]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "الان وضعیت مصرف CPU و RAM سیستم من چطوره؟"}
    ],
    tools=tools
)

# ببینیم مدل چی جواب داده:
message = response.choices[0].message
print("متن پیام:", message.content)
print("درخواست ابزار:", message.tool_calls)


متن پیام: None
درخواست ابزار: [ChatCompletionMessageFunctionToolCall(id='call_nJXTJLtFMmlpVgbWiUimn1pf', function=Function(arguments='{}', name='get_system_stats'), type='function')]


In [8]:
available_tools = {
    "get_system_stats": get_system_stats
}


In [9]:
# ۱. پیام مدل را به تاریخچه اضافه می‌کنیم
messages = [
    {"role": "user", "content": "الان وضعیت مصرف CPU و RAM  آیا نیاز هست نگران باشم؟سیستم من چطوره؟"},
    message  # همین پیامی که الان مدل فرستاد و توش tool_calls هست
]

# ۲. اجرای ابزار با پایتون
tool_call = message.tool_calls[0]
tool_name = tool_call.function.name
function_to_call = available_tools[tool_name]

# تابع را صدا می‌زنیم
tool_output = function_to_call()

# ۳. خروجی را با نقش tool به تاریخچه اضافه می‌کنیم
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": tool_output
})

# ۴. پیام نهایی را از مدل می‌خواهیم
final_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

print("\nپاسخ نهایی مدل:")
print(final_response.choices[0].message.content)



پاسخ نهایی مدل:
وضعیت مصرف CPU شما ۱۰٫۳ درصد و مصرف RAM شما ۵۰٫۲ درصد است. به طور کلی، این اعداد نشان‌دهنده بار نسبتاً مناسبی بر روی سیستم شما هستند. 

- مصرف CPU زیر ۲۰٪ عادی و سالم است، بنابراین نگرانی خاصی وجود ندارد.
- مصرف RAM شما نیز اگر زیر ۷۰٪ باشد معمولاً نگران کننده نیست، در حالی که ۵۰٫۲٪ هنوز در محدوده امن است.

از حیث مصرف منابع، سیستم شما خوب به نظر می‌رسد. اگر تجربه مشکلاتی در سرعت یا عملکرد دارید، ممکن است دلایل دیگری وجود داشته باشد، اما بر اساس این اطلاعات، نیازی به نگرانی نیست.


In [10]:
def run_agent(user_prompt):
    messages = [{"role": "user", "content": user_prompt}]
    max_turns = 5

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=messages,
            tools=tools,
        )

        message = response.choices[0].message
        messages.append(message)

        # اگر مدل درخواست ابزار نداد، یعنی پاسخ نهایی آماده است
        if not message.tool_calls:
            return message.content

        # اگر ابزار خواست، آن را اجرا می‌کنیم
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            function_to_call = available_tools[tool_name]

            # اجرای تابع واقعی
            tool_output = function_to_call()

            # برگرداندن خروجی ابزار به تاریخچه پیام‌ها
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_output,
                }
            )

    return "تعداد تلاش‌ها از حد مجاز گذشت."


In [11]:


print(run_agent("من حیلی دوست دارم روی ابونتو گیم داشته باشم ولی نمیدونم با وضعیت فعلی رم و سی پی یو میشه اینکارو کرد یا نه "))


وضعیت فعلی سیستم شما به این صورت است:
- مصرف CPU: حدود 16%
- مصرف RAM: حدود 51%

این یعنی نیمی از رم شما در حال استفاده است و پردازنده (CPU) خیلی شلوغ نیست. برای گیمینگ سبک یا بازی‌های ساده، به احتمال زیاد می‌توانید بدون مشکل بازی کنید. اما برای بازی‌های سنگین‌تر، مثل بازی‌های AAA با گرافیک بالا، باید بدانید دقیقاً چقدر رم و چه پردازنده‌ای دارید (مقدار کل رم مثلا 8GB یا 16GB و مدل CPU) و همچنین کارت گرافیک شما مهم است.

اگر اطلاعات بیشتری از سیستم بدهید (مثل مدل CPU، مقدار کل رم و نوع کارت گرافیک)، دقیق‌تر می‌توانم راهنمایی کنم و بازی‌های مناسب ابونتو را پیشنهاد بدهم. دوست داری راهنمای نصب استیم (Steam) روی ابونتو یا بازی‌های مناسب معرفی کنم؟


In [12]:

def get_process_stats(process_name: str) -> str:
    """دریافت درصد مصرف CPU و RAM یک پردازش خاص در اوبونتو"""
    found_processes = []
    
    # جستجو در تمام پردازش‌های در حال اجرا
    for proc in psutil.process_iter(['name', 'cpu_percent', 'memory_percent']):
        try:
            # اگر نام برنامه شامل متنی باشد که جستجو کردیم
            if process_name.lower() in proc.info['name'].lower():
                found_processes.append(
                    f"نام: {proc.info['name']} | CPU: {proc.info['cpu_percent']}% | RAM: {proc.info['memory_percent']:.1f}%"
                )
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue
            
    if found_processes:
        # نمایش حداکثر ۳ نتیجه پیدا شده
        return "\n".join(found_processes[:3])
    else:
        return f"هیچ پردازشی با نام '{process_name}' پیدا نشد."

In [13]:
Tools = [
    {
        "type": "function",
        "function": {
            "name": "get_process_stats",
            "description": "دریافت میزان مصرف پردازنده و رم یک برنامه یا پردازش خاص در سیستم",
            "parameters": {
                "type": "object",
                "properties": {
                    "process_name": {
                        "type": "string",
                        "description": "نام برنامه یا پردازشی که کاربر می‌خواهد وضعیت آن را بررسی کند، مثلا: python یا chrome"
                    }
                },
                "required": ["process_name"]
            }
        }
    }
]


In [14]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=Tools,
    tool_choice={"type": "function", "function": {"name": "get_process_stats"}} # این یعنی: «فقط از این ابزار استفاده کن»
)


In [15]:
tool_call = response.choices[0].message.tool_calls[0]
function_name = tool_call.function.name
# تبدیل رشته JSON ورودی‌ها به دیکشنری پایتون
function_args = json.loads(tool_call.function.arguments)

# ۲. اجرای واقعی تابع پایتون
if function_name == "get_process_stats":
    result = get_process_stats(process_name=function_args.get("process_name"))

# ۳. به‌روزرسانی تاریخچه پیام‌ها
messages = [
    {"role": "user", "content":" برنامه firefox چیه؟"},
    response.choices[0].message,  # پاسخ اول مدل (شامل tool_calls)
    {
        "role": "tool",
        "tool_call_id": tool_call.id,  # شناسه یکتای درخواست
        "content": result               # خروجی واقعی تابع پایتون
    }
]

# ۴. ارسال مجدد به مدل برای تولید پاسخ نهایی برای کاربر
final_response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages
)

print(final_response.choices[0].message.content)


فایرفاکس یک مرورگر وب آزاد و متن‌باز است که توسط شرکت موزیلا توسعه پیدا کرده است. این برنامه برای مرور و باز کردن صفحات وب به کار رفته و به خاطر قابلیت‌های امنیتی و حریم خصوصی برجسته‌ای که دارد، مشهور است. فایرفاکس قابلیت‌هایی مانند تب‌های مرورگری، مسدود کردن تبلیغات، حفاظت از رد پای دیجیتال کاربر، و افزونه‌های قابل نصب متعدد را فراهم می‌آورد. این مرورگر برای سیستم‌عامل‌های مختلفی از جمله ویندوز، مک، لینوکس، اندروید و iOS در دسترس است.


In [16]:
import json
import psutil
import time
from openai import OpenAI




client = OpenAI()
def get_process_stats(process_name: str):
    total_memory = 0
    total_cpu = 0
    found = False
    for proc in psutil.process_iter(['name', 'memory_info', 'cpu_percent']):
        try:
            if process_name.lower() in proc.info['name'].lower():
                found = True
                total_memory += proc.info['memory_info'].rss / (1024 * 1024)
                total_cpu += proc.info['cpu_percent']
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    if found:
        return f"برنامه {process_name}: مصرف رم {total_memory:.2f} MB - مصرف پردازنده {total_cpu}%"
    return f"برنامه‌ای با نام {process_name} پیدا نشد."

def get_disk_space(path: str = "/"):
    try:
        usage = psutil.disk_usage(path)
        total_gb = usage.total / (1024**3)
        free_gb = usage.free / (1024**3)
        percent = usage.percent
        return f"مسیر {path}: کل فضا {total_gb:.2f}GB | فضای آزاد {free_gb:.2f}GB | درصد مصرف {percent}%"
    except Exception as e:
        return f"خطا در بررسی مسیر {path}: {str(e)}"

def get_system_uptime():
    boot_time = psutil.boot_time()
    uptime_seconds = time.time() - boot_time
    hours = int(uptime_seconds // 3600)
    minutes = int((uptime_seconds % 3600) // 60)
    return f"سیستم به مدت {hours} ساعت و {minutes} دقیقه است که روشن است."


tools = [
    {
        "type": "function",
        "function": {
            "name": "get_process_stats",
            "description": "استخراج میزان مصرف رم و پردازنده یک برنامه مشخص در اوبونتو",
            "parameters": {
                "type": "object",
                "properties": {
                    "process_name": {"type": "string", "description": "نام برنامه مثلا firefox یا python3"}
                },
                "required": ["process_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_disk_space",
            "description": "بررسی میزان فضای کل و آزاد دیسک برای یک مسیر مشخص",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "مسیر مورد نظر در سیستم‌عامل مثل / یا /home"}
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_system_uptime",
            "description": "دریافت مدت زمانی که سیستم‌عامل اوبونتو روشن بوده است",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    }
]

available_functions = {
    "get_process_stats": get_process_stats,
    "get_disk_space": get_disk_space,
    "get_system_uptime": get_system_uptime
}


def run_agent(user_query: str):
    print(f"\n==================== سوال کاربر: '{user_query}' ====================")
    messages = [{"role": "user", "content": user_query}]
    

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    print("\n--- [Response اول از مدل (تصمیم‌گیری)] ---")
    print(response_message)
    
    if response_message.tool_calls:
        messages.append(response_message)
        
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"\n--- [اجرای تابع پایتون: {function_name} با ورودی {function_args}] ---")
            
            function_to_call = available_functions[function_name]
            tool_output = function_to_call(**function_args)
            
            print(f"خروجی تابع: {tool_output}")
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(tool_output)
            })
        

        final_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages
        )
        print("\n--- [Final Response (پاسخ نهایی مدل به کاربر)] ---")
        print(final_response.choices[0].message.content)
    else:
        print("\n--- [پاسخ مستقیم بدون ابزار] ---")
        print(response_message.content)


run_agent("چقدر فضای آزاد توی مسیر / داریم؟")



==================== سوال کاربر: 'چقدر فضای آزاد توی مسیر / داریم؟' ====================

--- [Response اول از مدل (تصمیم‌گیری)] ---
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_VSKqI3KoO0TEPksQMbGG9Ki6', function=Function(arguments='{"path":"/"}', name='get_disk_space'), type='function')])

--- [اجرای تابع پایتون: get_disk_space با ورودی {'path': '/'}] ---
خروجی تابع: مسیر /: کل فضا 936.79GB | فضای آزاد 868.84GB | درصد مصرف 2.3%

--- [Final Response (پاسخ نهایی مدل به کاربر)] ---
در مسیر /، کل فضای موجود 936.79 گیگابایت است که از این مقدار، 868.84 گیگابایت فضای آزاد دارید (2.3% از فضا مورد استفاده قرار گرفته است).
